# 03_limpieza_datos

Proyecto ARIMA / ARIMAX
Modelación epidemiológica con variables meteorológicas.

# Obtención de los datos epidemiológicos 

In [2]:
# Inicio del análisis
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np  
# Importar datos epidemiológicos 
ubicacion_epi_janis = r"C:\Users\usuario1\OneDrive - Universidad de Antioquia\UNIVERSIDAD DE ANTIOQUIA\Proyecto SAT Dengue\Bases de datos\Secretaria de salud\BD_DENGUE_2021-2025_CAUCASIA.xlsx"
ubicacion_epi_marco = r"C:\Users\marco\OneDrive - Universidad de Antioquia\Documentos\2_recursos_ensenanza\5_arima\6_datos_arima\2_datos_version_marzo_25_2026\epidemiologicos.xlsx"
df_epi=pd.read_excel(ubicacion_epi_marco)

# Importar datos meteorológicos 
ubicacion_meteo_janis = r"C:\Users\usuario1\OneDrive - Universidad de Antioquia\UNIVERSIDAD DE ANTIOQUIA\Proyecto SAT Dengue\Bases de datos\Datos meteorológicos\Datos_NS_2021-2025.xlsx"
ubicacion_meteo_marco = r"C:\Users\marco\OneDrive - Universidad de Antioquia\Documentos\2_recursos_ensenanza\5_arima\6_datos_arima\2_datos_version_marzo_25_2026\meteorologicos.xlsx"
df_meteo=pd.read_excel(ubicacion_meteo_marco)



In [3]:
df_epi.head(2)

,ini_sin_,semana,año,area_,localidad_,cen_pobla_,vereda_,bar_ver_,dir_res_,nmun_proce,nmun_resi
0,2021-01-21,3,2021,2,NO APLICA,CAUCASIA,NO APLICA,CAUCASIA,CR 39 E 48 C SUR 36,CAUCASIA,ENVIGADO
1,2021-02-08,6,2021,2,NO APLICA,CAUCASIA,NO APLICA,CAUCASIA,CL 29 A 42-99,CAUCASIA,BOGOTA


In [4]:
df_meteo.head(2)

,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,QV2M,RH2M,PRECTOTCORR,WS2M,WS2M_MAX,WS2M_MIN,ALLSKY_SFC_UV_INDEX
0,2021,3,28.64,35.09,23.84,17.89,75.41,4.03,0.14,0.32,0.07,2.48
1,2021,4,28.48,35.05,23.69,15.91,68.69,0.70,0.14,0.25,0.07,2.47


In [5]:
df_meteo.columns

Index(['YEAR', 'DOY', 'T2M', 'T2M_MAX', 'T2M_MIN', 'QV2M', 'RH2M',
       'PRECTOTCORR', 'WS2M', 'WS2M_MAX', 'WS2M_MIN', 'ALLSKY_SFC_UV_INDEX'],
      dtype='str')

In [6]:
df_epi.columns 

Index(['ini_sin_', 'semana', 'año', 'area_', 'localidad_', 'cen_pobla_',
       'vereda_', 'bar_ver_', 'dir_res_', 'nmun_proce', 'nmun_resi'],
      dtype='str')

# Remuestreo de los datos Epidemiológicos a frecuencia semanal 

In [ ]:
# remuestreo epi has 2024

In [ ]:
# remuestreo epi solo 2025

In [ ]:
# concatenación de los dos epi. 

In [8]:
import pandas as pd

# Asegurar formato datetime
df_epi['ini_sin_'] = pd.to_datetime(df_epi['ini_sin_'], errors='coerce')

# Eliminar filas con fechas nulas (opcional, pero recomendado)
df_epi = df_epi.dropna(subset=['ini_sin_'])

# Ordenar cronológicamente
df_epi = df_epi.sort_values("ini_sin_").reset_index(drop=True)

# --- Función para obtener semana epidemiológica (domingo a sábado) ---
def get_epidemiological_week(date):
    """
    Calcula la semana epidemiológica considerando que la semana 1 comienza
    el primer domingo del año.
    Retorna (año, número_de_semana)
    """
    # Encontrar el primer domingo del año
    year = date.year
    first_day = pd.Timestamp(f"{year}-01-01")
    
    # Días hasta el primer domingo (6 = domingo en weekday, donde lunes=0)
    days_until_sunday = (6 - first_day.weekday()) % 7
    first_sunday = first_day + pd.Timedelta(days=days_until_sunday)
    
    # Si la fecha es anterior al primer domingo, pertenece a la última semana del año anterior
    if date < first_sunday:
        # Calcular para el año anterior
        prev_year = year - 1
        prev_first_day = pd.Timestamp(f"{prev_year}-01-01")
        prev_days_until_sunday = (6 - prev_first_day.weekday()) % 7
        prev_first_sunday = prev_first_day + pd.Timedelta(days=prev_days_until_sunday)
        
        # Calcular semana en año anterior
        days_diff = (date - prev_first_sunday).days
        week_num = (days_diff // 7) + 1
        return prev_year, week_num
    else:
        # Calcular semana en año actual
        days_diff = (date - first_sunday).days
        week_num = (days_diff // 7) + 1
        return year, week_num

# Aplicar la función para obtener año y semana epidemiológica correctos
df_epi[["año_epi", "semana_epidemiologica"]] = df_epi["ini_sin_"].apply(
    lambda x: pd.Series(get_epidemiological_week(x))
)

# Agrupar casos por año y semana epidemiológica
df_casos_semana = (
    df_epi
    .groupby(["año_epi", "semana_epidemiologica"])
    .size()
    .reset_index(name="casos_dengue")
)

# Renombrar columna para mantener consistencia
df_casos_semana = df_casos_semana.rename(columns={"año_epi": "año"})

# Crear fecha correspondiente al DOMINGO de inicio de cada semana epidemiológica
def get_week_start_date(row):
    """Obtiene la fecha del domingo de inicio para una semana epidemiológica específica"""
    year = row["año"]
    week = row["semana_epidemiologica"]
    
    # Encontrar el primer domingo del año
    first_day = pd.Timestamp(f"{year}-01-01")
    days_until_sunday = (6 - first_day.weekday()) % 7
    first_sunday = first_day + pd.Timedelta(days=days_until_sunday)
    
    # Calcular fecha del domingo de la semana correspondiente
    week_start = first_sunday + pd.Timedelta(weeks=week-1)
    return week_start

df_casos_semana["fecha"] = df_casos_semana.apply(get_week_start_date, axis=1)

# --- CORRECCIÓN: Forzar inicio en la primera semana epidemiológica de 2021 ---
# Fecha de inicio: primer domingo de 2021 (3 de enero)
fecha_inicio = pd.Timestamp("2021-01-03")

# Fecha fin: última fecha en los datos
fecha_fin = df_casos_semana["fecha"].max()

# Crear un índice con todas las semanas (DOMINGOS) desde el 3 de enero de 2021
rango_completo = pd.date_range(start=fecha_inicio, end=fecha_fin, freq='W-SUN')

# Reindexar para incluir todas las semanas, rellenando con cero donde no hay casos
df_casos_semana = (
    df_casos_semana
    .set_index("fecha")
    .reindex(rango_completo, fill_value=0)
    .reset_index()
    .rename(columns={"index": "fecha"})
)

# Recalcular año y semana epidemiológica
resultados = df_casos_semana["fecha"].apply(
    lambda x: pd.Series(get_epidemiological_week(x), index=["año", "semana_epidemiologica"])
)

# Asignar los resultados al DataFrame
df_casos_semana[["año", "semana_epidemiologica"]] = resultados

# Seleccionar las variables epidemiológicas relevantes
columnas_seleccionadas = ["fecha", "semana_epidemiologica", "casos_dengue"]

df_casos_semana = df_casos_semana[columnas_seleccionadas]

# Convertir fecha en índice temporal
df_casos_semana = df_casos_semana.set_index("fecha").sort_index()
df_casos_semana 

,semana_epidemiologica,casos_dengue
fecha,,
2021-01-03,1,0
2021-01-10,2,0
2021-01-17,3,1
2021-01-24,4,0
2021-01-31,5,0
...,...,...
2025-11-30,48,8
2025-12-07,49,9
2025-12-14,50,7


In [9]:

# Verificar las primeras semanas
print("Primeras semanas epidemiológicas de 2021:")
print(df_casos_semana.head(10))

# Verificar específicamente la semana 1 de 2021
print("\nVerificación semana 1 de 2021 (debería ser 2021-01-03):")
print(df_casos_semana[(df_casos_semana.index.year == 2021) & 
                      (df_casos_semana["semana_epidemiologica"] == 1)])

Primeras semanas epidemiológicas de 2021:
            semana_epidemiologica  casos_dengue
fecha                                          
2021-01-03                      1             0
2021-01-10                      2             0
2021-01-17                      3             1
2021-01-24                      4             0
2021-01-31                      5             0
2021-02-07                      6             1
2021-02-14                      7             1
2021-02-21                      8             0
2021-02-28                      9             1
2021-03-07                     10             0

Verificación semana 1 de 2021 (debería ser 2021-01-03):
            semana_epidemiologica  casos_dengue
fecha                                          
2021-01-03                      1             0


In [10]:
df_casos_semana.columns 

Index(['semana_epidemiologica', 'casos_dengue'], dtype='str')

# Renombramiento de los atributos Meteorológicos 

In [11]:
# Renombrar columnas del dataframe nasa
df_meteo.rename(columns={
    'YEAR': 'año',
    'DOY': 'dia',
    'T2M': 'temp',
    'T2M_MAX': 'temp_max',
    'T2M_MIN': 'temp_min',
    'QV2M': 'hum_esp',
    'RH2M': 'hum_rel',
    'PRECTOTCORR': 'prec',
    'WS2M': 'vel_vi',
    'WS2M_MAX': 'vel_vi_max',
    'WS2M_MIN': 'vel_vi_min',
    'ALLSKY_SFC_UV_INDEX': 'uv'
}, inplace=True)
df_meteo.head(3)

,año,dia,temp,temp_max,temp_min,hum_esp,hum_rel,prec,vel_vi,vel_vi_max,vel_vi_min,uv
0,2021,3,28.64,35.09,23.84,17.89,75.41,4.03,0.14,0.32,0.07,2.48
1,2021,4,28.48,35.05,23.69,15.91,68.69,0.70,0.14,0.25,0.07,2.47
2,2021,5,27.84,34.96,22.71,15.16,68.47,0.62,0.16,0.34,0.03,2.30


# 5. Ordenar los datos meteorológicos cronológicamente en formato DD-MM-AAAA

In [12]:
# Asegurar que las columnas 'año' y 'dia' sean numéricas 
df_meteo["año"] = df_meteo["año"].astype(int)
df_meteo["dia"] = df_meteo["dia"].astype(int)

# Crear la columna 'fecha' combinando año y día del año 
df_meteo["fecha"] = pd.to_datetime(df_meteo["año"].astype(str), format="%Y") + pd.to_timedelta(df_meteo["dia"] - 1, unit="D")

# Eliminar las columnas originales 'año' y 'dia'
df_meteo = df_meteo.drop(columns=[ "dia"])

# Reordenar las columnas para que 'fecha' quede al inicio 
cols = ["fecha"] + [c for c in df_meteo.columns if c != "fecha"]
df_meteo = df_meteo[cols]

# Ordenar cronológicamente 
df_meteo_crono = df_meteo.sort_values("fecha").reset_index(drop=True)

df_meteo_crono.head()


,fecha,año,temp,temp_max,temp_min,hum_esp,hum_rel,prec,vel_vi,vel_vi_max,vel_vi_min,uv
0,2021-01-03,2021,28.64,35.09,23.84,17.89,75.41,4.03,0.14,0.32,0.07,2.48
1,2021-01-04,2021,28.48,35.05,23.69,15.91,68.69,0.70,0.14,0.25,0.07,2.47
2,2021-01-05,2021,27.84,34.96,22.71,15.16,68.47,0.62,0.16,0.34,0.03,2.30
3,2021-01-06,2021,28.58,35.31,23.50,15.64,67.09,0.00,0.11,0.22,0.04,2.34
4,2021-01-07,2021,28.52,34.44,23.73,15.94,67.92,0.04,0.11,0.22,0.05,2.03


# Remuestrear los datos meteorológicos a frecuencia de semana epidemiológica 

In [ ]:
# remuestreo meteo hasta 2024

In [ ]:
# remuestre meto solo 2025

In [ ]:
# Concatenar estas dos meteo

In [13]:
import pandas as pd

# --- CORRECCIÓN: Función para obtener semana epidemiológica (domingo a sábado) ---
def get_epidemiological_week(date):
    """
    Calcula la semana epidemiológica considerando que la semana 1 comienza
    el primer domingo del año.
    Retorna (año, número_de_semana)
    """
    # Encontrar el primer domingo del año
    year = date.year
    first_day = pd.Timestamp(f"{year}-01-01")
    
    # Días hasta el primer domingo (6 = domingo en weekday, donde lunes=0)
    days_until_sunday = (6 - first_day.weekday()) % 7
    first_sunday = first_day + pd.Timedelta(days=days_until_sunday)
    
    # Si la fecha es anterior al primer domingo, pertenece a la última semana del año anterior
    if date < first_sunday:
        # Calcular para el año anterior
        prev_year = year - 1
        prev_first_day = pd.Timestamp(f"{prev_year}-01-01")
        prev_days_until_sunday = (6 - prev_first_day.weekday()) % 7
        prev_first_sunday = prev_first_day + pd.Timedelta(days=prev_days_until_sunday)
        
        # Calcular semana en año anterior
        days_diff = (date - prev_first_sunday).days
        week_num = (days_diff // 7) + 1
        return prev_year, week_num
    else:
        # Calcular semana en año actual
        days_diff = (date - first_sunday).days
        week_num = (days_diff // 7) + 1
        return year, week_num

# 1. Asegurar que la columna 'fecha' es tipo datetime
df_meteo_crono["fecha"] = pd.to_datetime(df_meteo_crono["fecha"])

# 2. Crear columna binaria: día con lluvia (1 si Prec ≥ 1 mm)
df_meteo_crono["lluvia"] = (df_meteo_crono["prec"] >= 1).astype(int)

# 3. Definir la fecha como índice temporal
df_meteo_crono = df_meteo_crono.set_index("fecha")

# 4. Remuestrear de diario a semanal (DOMINGO a sábado - usando W-SAT para que el resultado sea sábado)
df_meteo_crono_semanal = df_meteo_crono.resample("W-SAT").agg({
    "temp": "mean",
    "temp_max": "mean",
    "temp_min": "mean",
    "hum_esp": "mean",
    "hum_rel": "mean",
    "prec": "sum",
    "lluvia": "sum",
    "vel_vi": "mean",
    "vel_vi_max": "mean",
    "vel_vi_min": "mean",
    "uv": "mean"
})

# 5. Renombrar variable
df_meteo_crono_semanal = df_meteo_crono_semanal.rename(columns={"lluvia": "dias_lluvia"})

# 6. Convertir índice en columna
df_meteo_crono_semanal = df_meteo_crono_semanal.reset_index()

# --- CORRECCIÓN: Calcular semana epidemiológica correcta ---
# Aplicar la función para obtener año y semana epidemiológica correctos
df_meteo_crono_semanal[["año", "semana_epidemiologica"]] = df_meteo_crono_semanal["fecha"].apply(
    lambda x: pd.Series(get_epidemiological_week(x))
)

# --- CORRECCIÓN: Ajustar fecha para que sea el DOMINGO de inicio de la semana ---
# Restar 6 días para ir del sábado (resultado de resample) al domingo anterior
df_meteo_crono_semanal["fecha"] = df_meteo_crono_semanal["fecha"] - pd.Timedelta(days=6)

# --- CORRECCIÓN: Forzar inicio en la primera semana epidemiológica de 2021 ---
# Fecha de inicio: primer domingo de 2021 (3 de enero)
fecha_inicio = pd.Timestamp("2021-01-03")

# Filtrar solo datos desde fecha_inicio en adelante
df_meteo_crono_semanal = df_meteo_crono_semanal[df_meteo_crono_semanal["fecha"] >= fecha_inicio]

# Obtener fecha fin
fecha_fin = df_meteo_crono_semanal["fecha"].max()

# Crear un índice con todas las semanas (DOMINGOS) desde el 3 de enero de 2021
rango_completo = pd.date_range(start=fecha_inicio, end=fecha_fin, freq='W-SUN')

# Reindexar para incluir todas las semanas
df_meteo_crono_semanal = (
    df_meteo_crono_semanal
    .set_index("fecha")
    .reindex(rango_completo)
    .reset_index()
    .rename(columns={"index": "fecha"})
)

# Recalcular año y semana epidemiológica para las nuevas filas (si es necesario)
# Interpolar valores faltantes (opcional - puedes elegir diferentes métodos)
# Para variables continuas: interpolación lineal
vars_continuas = ["temp", "temp_max", "temp_min", "hum_esp", "hum_rel", "vel_vi", 
                  "vel_vi_max", "vel_vi_min", "uv"]
df_meteo_crono_semanal[vars_continuas] = df_meteo_crono_semanal[vars_continuas].interpolate(method='linear')

# Para variables de conteo: rellenar con 0 o usar interpolación más suave
df_meteo_crono_semanal["prec"] = df_meteo_crono_semanal["prec"].fillna(0)
df_meteo_crono_semanal["dias_lluvia"] = df_meteo_crono_semanal["dias_lluvia"].fillna(0).round(0)

# Recalcular año y semana epidemiológica
resultados = df_meteo_crono_semanal["fecha"].apply(
    lambda x: pd.Series(get_epidemiological_week(x), index=["año", "semana_epidemiologica"])
)
df_meteo_crono_semanal[["año", "semana_epidemiologica"]] = resultados

# 11. Ordenar columnas
columnas_base = ["fecha", "año", "semana_epidemiologica"]
columnas_restantes = [col for col in df_meteo_crono_semanal.columns 
                     if col not in columnas_base]

df_meteo_crono_semanal = df_meteo_crono_semanal[columnas_base + columnas_restantes]

# 12. Dejar fecha como índice
df_meteo_crono_semanal = df_meteo_crono_semanal.set_index("fecha").sort_index()

# Verificar las primeras semanas
print("Primeras semanas meteorológicas de 2021:")
print(df_meteo_crono_semanal.tail(10))

# Verificar específicamente la semana 1 de 2021
print("\nVerificación semana 1 de 2021 (debería ser 2021-01-03):")
print(df_meteo_crono_semanal[(df_meteo_crono_semanal.index.year == 2021) & 
                             (df_meteo_crono_semanal["semana_epidemiologica"] == 1)])

Primeras semanas meteorológicas de 2021:
             año  semana_epidemiologica       temp   temp_max   temp_min  \
fecha                                                                      
2025-10-26  2025                     43  26.112857  28.492857  24.381429   
2025-11-02  2025                     44  26.088571  28.888571  23.961429   
2025-11-09  2025                     45  25.895714  28.344286  24.254286   
2025-11-16  2025                     46  25.630000  28.507143  23.415714   
2025-11-23  2025                     47  25.927143  28.895714  23.790000   
2025-11-30  2025                     48  25.527143  28.834286  23.198571   
2025-12-07  2025                     49  26.082857  29.091429  23.785714   
2025-12-14  2025                     50  26.020000  29.442857  23.658571   
2025-12-21  2025                     51  25.988571  30.090000  23.008571   
2025-12-28  2025                     52  26.085714  29.490000  23.778571   

              hum_esp    hum_rel   prec  dias_

In [14]:
df_meteo_crono_semanal.columns

Index(['año', 'semana_epidemiologica', 'temp', 'temp_max', 'temp_min',
       'hum_esp', 'hum_rel', 'prec', 'dias_lluvia', 'vel_vi', 'vel_vi_max',
       'vel_vi_min', 'uv'],
      dtype='str')

# Unir las bases de datos epidemiológicas con la base de datos meteorológica de acuerdo al criterop año semana epidemiológica 


In [15]:


# Unir por índice (fecha)
df_epi_meteo = df_meteo_crono_semanal.join(df_casos_semana[["casos_dengue"]], how="left")



In [16]:
df_epi_meteo.head(3)

,año,semana_epidemiologica,temp,temp_max,temp_min,hum_esp,hum_rel,prec,dias_lluvia,vel_vi,vel_vi_max,vel_vi_min,uv,casos_dengue
fecha,,,,,,,,,,,,,,
2021-01-03,2021,1,28.252857,34.200000,23.832857,16.308571,70.508571,5.72,1,0.121429,0.244286,0.047143,2.222857,0
2021-01-10,2021,2,28.687143,34.910000,24.195714,17.318571,72.885714,19.15,5,0.117143,0.208571,0.038571,2.254286,0
2021-01-17,2021,3,29.592857,36.372857,24.090000,16.112857,65.122857,0.77,0,0.124286,0.225714,0.045714,2.420000,1


In [17]:


df_epi_meteo.tail(3)

,año,semana_epidemiologica,temp,temp_max,temp_min,hum_esp,hum_rel,prec,dias_lluvia,vel_vi,vel_vi_max,vel_vi_min,uv,casos_dengue
fecha,,,,,,,,,,,,,,
2025-12-14,2025,50,26.020000,29.442857,23.658571,18.935714,89.791429,30.61,6,0.168571,0.447143,0.034286,2.067143,7
2025-12-21,2025,51,25.988571,30.090000,23.008571,17.994286,85.884286,2.51,1,0.185714,0.452857,0.018571,1.995714,5
2025-12-28,2025,52,26.085714,29.490000,23.778571,19.070000,90.110000,14.39,3,0.131429,0.281429,0.025714,1.842500,9


In [18]:
df_epi_meteo[['casos_dengue']]

,casos_dengue
fecha,
2021-01-03,0
2021-01-10,0
2021-01-17,1
2021-01-24,0
2021-01-31,0
...,...
2025-11-30,8
2025-12-07,9
2025-12-14,7


# Convertir el DataFrame df_epi_meteo en un .excel

In [22]:
# convertir este dataset fusionado en archivo .xlsx para que pueda ser utilizado en el análisis de series de tiempo con ARIMA
df_epi_meteo.to_excel(r"C:\Users\marco\OneDrive - Universidad de Antioquia\Documentos\2_recursos_ensenanza\5_arima\6_datos_arima\3_base_datos_consolidada\datos_fusionados_semanales.xlsx", index=True) 

Hasta aquí hemos logrado la estructuración de la base de datos epidemiológica y meteorológica.

Tarea de resumir esta limpieza, **remuestreo** y fusion de datos, en una sola diapositiva. 